In [1]:
%load_ext autoreload
%autoreload 2 --print

In [2]:
import torch as th
import torch.jit as jit
import copy
import numpy as np
import dill
from collections import namedtuple
from torch import optim
from torch import nn
from dynrn.rnntasks import (
    DriscollTasks,
    DriscollPlots,
    period_start_mask,
    periwindows,
    periperiod_sliced,
    split_trials,
    extract_trial_data,
    apply_to_trial_groups,
    nanmean_with_req,
    split_trials_driscoll,
)
from dynrn.predictors import (
    activity_dataset,
    save_dsn,
    load_dsn,
    discounted_sums,
    MultiBlockActivityDataset,
)
import dynrn.basic_rnns as rnns
from dynrn.basic_rnns import find_hash
from dynrn.viz import dynamics as vd
import scipy.stats
from cmap import Colormap
from scipy.stats import uniform, norm
from datetime import datetime
from mplutil import util as vu
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
import tqdm
from numpy import linalg as la
import os
import joblib as jl
import time
import seaborn as sns

In [3]:
from dynrn.viz import styles
from dynrn.viz.styles import getc
t20 = lambda x: getc(f"seaborn:tab20{x}")
colors, plotter = styles.init_plt(
    Path('../plots/notebook/multi-pred-failure').resolve(),
    fmt = 'pdf', display=False)
plot_root = Path(plotter.plot_dir)

period_colors = {
    "iti": t20("b:5"),
    "context": t20("c:1"),
    "stim": t20("b:2"),
    "memory": t20("b:14"),
    "response": t20("b:10")
}

In [4]:
# cuda setup
device = th.device('cuda' if th.cuda.is_available() else 'cpu')
cpu = th.device('cpu' if th.cuda.is_available() else 'cpu')
print(device.type)

cpu


### Load predictions

In [5]:
root_dir = "/Users/kaifox/projects/loop/dynrn/data"
dsn_hash = '7c61e2'

# load discounted sum network pointed to by hash
dsn_path = find_hash(root_dir, dsn_hash, ".pt")
dsn_path = Path(str(dsn_path)[:-3])  # remove .pt
final_dsn, dsn_ckpts, traindata = rnns.load_rnn(dsn_path, device=device)

# Load associated activity dataset
# act_data_hash = traindata["act_hash"]
act_data_hash = '7c6042'
act_data = dill.load(open(find_hash(root_dir, act_data_hash, ".dil"), "rb"))

# Select main variables from activity dataset
test_data: MultiBlockActivityDataset = act_data["test"]
test_blocks = DriscollTasks.split_dataset(test_data)
cumulant_fn = lambda x: x[:, 1:]
gamma = traindata["gamma"]

# set plotting directory
dsn_name = dsn_path.name
nb_plot_dir = Path('../plots/notebook').resolve()
plotter.plot_dir = nb_plot_dir / 'multi-pred-failure' / dsn_name
plotter.plot_dir.mkdir(exist_ok=True)

In [6]:
# discounted sum predictions
x = th.tensor(test_data["activity"], dtype=th.float32, device=device)
preds = final_dsn.to(device)(x).cpu().detach().numpy()

# extraction of cumulant from sum
cumul_gt = cumulant_fn(test_data["activity"])
cumul_pr = preds[:, :-1] - gamma * preds[:, 1:]

# timepoint-wise loss
loss_series = la.norm(cumul_gt - cumul_pr, axis=-1)

### Loss relative to trial start

In [7]:
# indexing: period_block_loss[i_block][period_number]: DataFrame[rel_time, number, 0, 1, 2, ...]
# where column "0" indexes last dimension of `loss_series`

period_block_loss = {}
for i_block, slc in enumerate(test_data['block_slices']):
    period_block_loss[i_block] = periperiod_sliced(
        loss_series[slc, :, None],
        test_data['periods'][slc, 1:].astype('int')
    )

In [8]:
# find task-wise, period-wise PCs

# shorthand for pca.transform on high-dim array
_pct = lambda pca, x: pca.transform(x.reshape(-1, x.shape[-1])).reshape(
    x.shape[:-1] + (pca.n_components_,)
)

def _calc(task, slc):
    # list of trials, where each trial is dict of lists of arrays
    # ex: trials[0]['period'] = [[4, 0, 1], [0, 1, 2], ..., [3, 4, 0]]
    # and trials[0]['gt'] has same list strucutre, but with period indices
    # replaced by `cumulpc_gt` data from the corresponding period
    trials_fulldim = split_trials(
        {"gt": cumul_gt[slc]},
        test_data['periods'][slc].astype("int")[:, 1:],
    )
    trials_fulldim = list(
        filter(  # filter out end-of-session ITI
            (lambda t: len(t["period"]) >= 5), trials_fulldim
        )
    )
    # fit pca for activity during each period for visualization
    _npr = task.n_period
    period_cumul_gt = [
        np.concatenate([t["gt"][i] for t in trials_fulldim]) for i in range(_npr)
    ]
    period_cumul_pca = [PCA(n_components=3).fit(h) for h in period_cumul_gt]
    full_pca = PCA(n_components=3).fit(cumul_gt.reshape(-1, cumul_gt.shape[-1]))

    # gt and predicted cumulants and predicted future sum, projected onto each PC axis
    # shape: (n_sessions, session_length, 2, n_period)
    cumulpc_gt = np.stack(
        [_pct(pca, cumul_gt[slc]) for pca in period_cumul_pca],
        axis=-1,
    )
    cumulpc_pr = np.stack(
        [_pct(pca, cumul_pr[slc]) for pca in period_cumul_pca],
        axis=-1,
    )

    cumulfpc_gt = np.stack(
        [_pct(full_pca, cumul_gt[slc]) for pca in period_cumul_pca],
        axis=-1,
    )
    cumulfpc_pr = np.stack(
        [_pct(full_pca, cumul_pr[slc]) for pca in period_cumul_pca],
        axis=-1,
    )

    return period_cumul_pca, cumulpc_gt, cumulpc_pr

period_pca, cumulpc_gt, cumulpc_pr = {}, {}, {}
for i, (_t, _s) in enumerate(zip(test_data['block_tasks'], test_data['block_slices'])):
    _p, _g, _r = _calc(_t, _s)
    period_pca[i] = _p
    cumulpc_gt[i] = _g
    cumulpc_pr[i] = _r

In [9]:
def _calc(block, task, slc):
    # list of trials, where each trial is dict of lists of arrays
    # ex: trials['period'][0] = [[4, 0, 1], [0, 1, 2], ..., [3, 4, 0]]
    # and trials['pc_gt'][0] has same list strucutre, but with period indices
    # replaced by `cumulpc_gt` data from the corresponding period
    trials = split_trials_driscoll(
        {
            "pc_gt": cumulpc_gt[block],
            "pc_pr": cumulpc_pr[block],
            "stim": test_data['stimuli'][slc],
        },
        test_blocks[block],
        window=1,
    )

    # find stim angle for each trial and group by it
    trial_angles, trial_angle_colors, trial_groups = extract_trial_data(
        lambda x: np.arctan2(x[:, 3], x[:, 2]).mean(),
        trials,
        window = 1,
        cmap = Colormap("matlab:cool"),
        color_range = (0, np.pi / 2),
        n_clusters = 20
    )
    trial_group_angles = np.array([trial_angles[group].mean() for group in trial_groups])
    trial_group_angle_colors = Colormap("matlab:cool")(trial_group_angles / np.pi * 4)

    return trials, trial_angle_colors, trial_group_angle_colors


trials, trial_colors, avg_trial_colors = {}, {}, {}
for _i, (_t, _s) in enumerate(zip(test_data['block_tasks'], test_data['block_slices'])):
    _t, _tc, _atc = _calc(_i, _t, _s)
    trials[_i] = _t
    trial_colors[_i] = _tc
    # avg_trials[_i] = _at
    avg_trial_colors[_i] = _atc

In [35]:
_dataset = dill.load(open(find_hash(root_dir, act_data['dataset_hash'], ".dil"), "rb"))
_task = dill.load(open(find_hash(root_dir, _dataset['task_hash'], ".dil"), "rb"))
_task[list(_task.keys())[0]]['iti'].exp.args

Reloading 'dynrn.rnntasks'.


()